# 06 — Entraînement du surrogate model

Objectif : entraîner un modèle LightGBM qui prédit le niveau de bruit (dB)
à partir des **features de morphologie urbaine du papier** (notebook 04).

C'est notre extension du papier : eux montrent que la morphologie est corrélée au SPL,
nous on entraîne un modèle prédictif dessus.

**Input** : `data/processed/sunbird_morphology.parquet` (notebook 04)
**Output** : `outputs/models/surrogate_lgbm.pkl`

Ensuite ce modèle sera calibré sur nos mesures Hanoï (notebook 07).

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib

df = pd.read_parquet('../data/processed/sunbird_morphology.parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['is_weekend'] = df['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)
print(f'{len(df)} exemples chargés')
df.head()

In [ ]:
# Features = les métriques de morphologie du papier + temps
FEATURES = ['building_density_km2', 'road_density_km_km2', 'intersection_count',
            'dist_road_m', 'hour', 'is_weekend']
TARGET   = 'noise_measurement'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    subsample=0.8,
    random_state=42,
    verbose=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2   = r2_score(y_test, y_pred)

print(f'MAE  : {mae:.2f} dB')
print(f'RMSE : {rmse:.2f} dB')
print(f'R²   : {r2:.3f}')

In [ ]:
# Importance des features
lgb.plot_importance(model, figsize=(8, 4), title='Feature importance')
plt.tight_layout()
plt.savefig('../outputs/maps/feature_importance.png', dpi=150)
plt.show()

# Prédit vs réel
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.3, s=10)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax.set_xlabel('Mesuré (dB)')
ax.set_ylabel('Prédit (dB)')
ax.set_title(f'Surrogate model — R²={r2:.3f}')
plt.tight_layout()
plt.savefig('../outputs/maps/pred_vs_real.png', dpi=150)
plt.show()

In [ ]:
# Sauvegarde le modèle pré-entraîné
joblib.dump(model, '../outputs/models/surrogate_lgbm.pkl')
print('Modèle sauvegardé dans outputs/models/surrogate_lgbm.pkl')